# 03 — VAE: detección de anomalías viales

Este notebook **no entrena**: lee los resultados generados por

```powershell
python -m src.data.cache_images --size 128
python -m src.models.train_vae --config configs/vae/vae_base.yaml
```

El VAE se entrena solo con imágenes de `train` **sin daño anotado** (`has_any_damage == 0`). El checkpoint se elige
con la pérdida (β = 1) en las imágenes sin daño de `val`, sin usar etiquetas. Hay dos scores de anomalía:

- **global**: error cuadrático medio de toda la imagen;
- **patch**: máximo del error promediado en parches de 16×16 (pensado para baches pequeños).

Para cada score, el threshold maximiza F1 de D40 en `val`; el score principal es el de mayor PR-AUC en `val`.
Todo queda fijo para `test_id`, `test_ood` (Norway) y `test_perspective` (China_Drone).

Cada corrida guarda en `reports/experiments/vae/<experiment_id>/` su `config.yaml`, `history.csv`, `metrics.json`
y `predictions.csv` (contrato de 7 columnas + `y_score_global`, `y_score_patch`, `group`, `country`).
Las corridas `_limitN` son pruebas y se ignoran.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS = PROJECT_ROOT / "reports" / "experiments" / "vae"
SPLITS = ["val", "test_id", "test_ood", "test_perspective"]
GROUPS = ["no_damage", "other", "pothole"]

runs = {}
for folder in sorted(RESULTS.glob("*")) if RESULTS.exists() else []:
    if (folder / "metrics.json").exists() and "_limit" not in folder.name:
        runs[folder.name] = dict(
            config=yaml.safe_load((folder / "config.yaml").read_text(encoding="utf-8")),
            metrics=json.loads((folder / "metrics.json").read_text(encoding="utf-8")),
            history=pd.read_csv(folder / "history.csv"),
            predictions=pd.read_csv(folder / "predictions.csv"),
        )
print("Corridas encontradas:", list(runs) or "ninguna todavía")

## 1. Configuración registrada

In [ ]:
keys = ["latent", "image_size", "batch_size", "epochs", "lr", "patience", "warmup_epochs", "patch", "optimizer",
        "best_epoch", "train_images", "val_no_damage_images", "image_source", "seed", "dataset_version", "splits_sha256"]
pd.DataFrame({name: {k: run["config"].get(k) for k in keys} for name, run in runs.items()})

## 2. Curvas de entrenamiento

La pérdida de `val` es solo sobre imágenes sin daño y siempre con β = 1. Mientras β < 1 (warmup) no se elige
checkpoint. La PR-AUC de `val` se registra por época pero **no** interviene en la selección.

In [ ]:
for name, run in runs.items():
    h = run["history"]
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
    axes[0].plot(h.epoch, h.train_loss, label="train")
    axes[0].plot(h.epoch, h.val_loss, "--", label="val (sin daño, β=1)")
    axes[1].plot(h.epoch, h.train_recon, label="train")
    axes[1].plot(h.epoch, h.val_recon, "--", label="val")
    axes[2].plot(h.epoch, h.train_kl, label="train")
    axes[2].plot(h.epoch, h.val_kl, "--", label="val")
    axes[2].plot(h.epoch, h.beta * h.train_kl.max(), ":", color="gray", label="β (escalado)")
    axes[3].plot(h.epoch, h.val_pr_auc_global, label="global")
    axes[3].plot(h.epoch, h.val_pr_auc_patch, label="patch")
    axes[3].axvline(run["metrics"]["best_epoch"], color="k", lw=0.8, label="checkpoint")
    for ax, title in zip(axes, ["Pérdida total", "Reconstrucción", "KL", "PR-AUC D40 en val"]):
        ax.set(title=title, xlabel="epoch")
        ax.legend(fontsize=8)
    fig.suptitle(name)
    plt.tight_layout()

## 3. Métricas D40 por split

`y_score` del CSV es el score principal. La PR-AUC debe leerse junto a la prevalencia, que es la PR-AUC esperada
de un clasificador aleatorio y cambia entre splits.

In [ ]:
rows = []
for name, run in runs.items():
    m = run["metrics"]
    print(f"{name}: score principal = {m['primary_score']}, thresholds = {m['thresholds']}, "
          f"PR-AUC val = {m['val_pr_auc']}")
    for split, by_score in m["splits"].items():
        for score, values in by_score.items():
            rows.append(dict(experiment=name, split=split, score=score, n=values["n"],
                             prevalence=values["prevalence"], precision=values["precision"],
                             recall=values["recall"], f1=values["f1"], pr_auc=values["pr_auc"]))
table = pd.DataFrame(rows)
table.pivot(index=["experiment", "score"], columns="split", values=["prevalence", "pr_auc", "f1"]).reindex(
    columns=SPLITS, level=1).round(4) if not table.empty else "Sin resultados todavía"

## 4. ¿Detecta baches o cualquier daño?

Distribución del score principal por grupo. `flagged_rate` es la proporción sobre el threshold: en
`no_damage` y `other` son falsos positivos, en `pothole` es el recall. Si `other` se marca casi tanto como
`pothole`, el VAE responde a daño vial en general y no a baches en particular.

In [ ]:
for name, run in runs.items():
    primary = run["metrics"]["primary_score"]
    rows = [dict(split=split, group=group, **values)
            for split, report in run["metrics"]["breakdown"].items()
            for group, values in report[primary]["groups"].items()]
    display(pd.DataFrame(rows).pivot(index="group", columns="split", values=["n", "median", "flagged_rate"])
            .reindex(columns=SPLITS, level=1).reindex(GROUPS).round(4))

In [ ]:
for name, run in runs.items():
    primary = run["metrics"]["primary_score"]
    rows = [dict(split=split, pair=pair, **values)
            for split, report in run["metrics"]["breakdown"].items()
            for pair, values in report[primary]["pairwise_auroc"].items()]
    display(pd.DataFrame(rows).pivot(index="pair", columns="split", values="auroc").reindex(columns=SPLITS).round(4))

In [ ]:
for name, run in runs.items():
    predictions = run["predictions"]
    fig, axes = plt.subplots(1, len(SPLITS), figsize=(4.5 * len(SPLITS), 3.4), sharey=False)
    for ax, split in zip(axes, SPLITS):
        part = predictions[predictions.split == split]
        for group in GROUPS:
            values = part.loc[part.group == group, "y_score"]
            if len(values):
                ax.hist(values, bins=50, density=True, histtype="step", label=f"{group} (n={len(values)})")
        ax.axvline(part.threshold.iloc[0], color="k", lw=0.8)
        ax.set(title=split, xlabel=f"score ({run['metrics']['primary_score']})")
        ax.legend(fontsize=7)
    fig.suptitle(f"{name}: distribución del score por grupo (línea = threshold de val)")
    plt.tight_layout()

## 5. Confounding por país

Las imágenes sin daño de `train` y `val` vienen solo de Czech, India y Japan; United_States y China_MotorBike no
tienen ninguna. Un score alto en esos países puede deberse al dominio y no al daño, así que la comparación limpia
es **dentro de cada país** que tiene los dos grupos.

In [ ]:
for name, run in runs.items():
    primary = run["metrics"]["primary_score"]
    rows = []
    for split in ["val", "test_id"]:
        for country, values in run["metrics"]["breakdown"][split][primary]["by_country"].items():
            rows.append(dict(split=split, country=country, **{f"n_{g}": values["n"][g] for g in GROUPS},
                             pr_auc_D40=values["pr_auc_D40"],
                             auroc_no_damage_vs_pothole=values["pairwise_auroc"]["no_damage_vs_pothole"]["auroc"],
                             auroc_other_vs_pothole=values["pairwise_auroc"]["other_vs_pothole"]["auroc"]))
    display(pd.DataFrame(rows).set_index(["split", "country"]).round(4))

## 6. ¿Con qué otros daños se confunde?

Imágenes **sin D40**, separadas por cada clase anotada. Una imagen con varias clases cuenta en cada una.
`flagged_rate` = falsos positivos con el threshold de `val`.

In [ ]:
for name, run in runs.items():
    primary = run["metrics"]["primary_score"]
    rows = [dict(split=split, damage=damage, **values)
            for split, report in run["metrics"]["breakdown"].items()
            for damage, values in report[primary]["other_damage_classes"].items()]
    frame = pd.DataFrame(rows)
    display(frame.pivot(index="damage", columns="split", values=["n", "flagged_rate"])
            .reindex(columns=SPLITS, level=1).round(4))

## 7. Reconstrucciones y mapas de error

Ejemplos de `val` con el checkpoint elegido: original, reconstrucción y error por píxel. En las imágenes con D40
se dibujan las cajas de `annotations.csv` escaladas a 128×128. El mapa de error no es una segmentación.

In [ ]:
import sys

import torch
from matplotlib.patches import Rectangle

sys.path.insert(0, str(PROJECT_ROOT))  # para importar src desde notebooks/
from src.data.cache_images import cache_root, cached_frame
from src.data.dataset import DATASET_ROOT, PotholeDataset, build_transforms, load_split_frame
from src.models.vae import ConvVAE, error_map

annotations = pd.read_csv(PROJECT_ROOT / "data" / "manifests" / "annotations.csv")
frame = load_split_frame(["val"]).set_index("sample_id")
for name, run in runs.items():
    config, predictions = run["config"], run["predictions"]
    checkpoint = PROJECT_ROOT / "checkpoints" / "vae" / name / "best.pt"
    if not checkpoint.exists():
        print(f"{name}: falta {checkpoint.relative_to(PROJECT_ROOT)} (los checkpoints no van a Git)")
        continue
    model = ConvVAE(latent=config["latent"], image_size=config["image_size"])
    model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True)["model"])
    model.eval()
    val = predictions[predictions.split == "val"]
    # los dos con mayor score de cada grupo, para ver que marca el modelo
    picks = val.sort_values("y_score", ascending=False).groupby("group").head(2).sort_values("group")
    rows = frame.loc[picks.sample_id].reset_index()
    root = cache_root(config["image_size"]) if config["use_cache"] else DATASET_ROOT
    images = cached_frame(rows) if config["use_cache"] else rows
    dataset = PotholeDataset(images, build_transforms(size=config["image_size"], normalize=False), root)
    fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
    for i, (_, row) in enumerate(rows.iterrows()):
        image = dataset[i][0].unsqueeze(0)
        with torch.no_grad():
            recon = model(image)[0]
        errors = error_map(image, recon)[0, 0]
        score = picks.set_index("sample_id").loc[row.sample_id]
        axes[i, 0].imshow(image[0].permute(1, 2, 0))
        axes[i, 1].imshow(recon[0].permute(1, 2, 0).clamp(0, 1))
        axes[i, 2].imshow(errors, cmap="inferno")
        scale_x, scale_y = config["image_size"] / row.width, config["image_size"] / row.height
        for box in annotations[(annotations.sample_id == row.sample_id) & (annotations["class"] == "D40")].itertuples():
            for ax in (axes[i, 0], axes[i, 2]):
                ax.add_patch(Rectangle((box.xmin * scale_x, box.ymin * scale_y), box.bbox_width * scale_x,
                                       box.bbox_height * scale_y, fill=False, edgecolor="cyan", lw=1.2))
        axes[i, 0].set_title(f"{score.group} · {row.country}", fontsize=8)
        axes[i, 2].set_title(f"score {score.y_score:.4f}", fontsize=8)
        for ax in axes[i]:
            ax.axis("off")
    fig.suptitle(f"{name}: original · reconstrucción · error")
    plt.tight_layout()

## 8. Latencia (batch 1, solo forward con z = μ)

In [ ]:
pd.DataFrame({name: run["metrics"]["latency"] for name, run in runs.items()}).T

## 9. Comparación con ResNet

Misma métrica (PR-AUC de D40) sobre los mismos splits. Solo aparece si ya existen resultados de ResNet.

In [ ]:
resnet_rows = []
for folder in sorted((PROJECT_ROOT / "reports" / "experiments" / "resnet").glob("*")):
    if (folder / "metrics.json").exists() and "_limit" not in folder.name:
        metrics = json.loads((folder / "metrics.json").read_text(encoding="utf-8"))
        resnet_rows += [dict(model=f"resnet · {folder.name}", split=split, pr_auc=values["pr_auc"], f1=values["f1"])
                        for split, values in metrics["splits"].items()]
vae_rows = [dict(model=f"vae · {name} ({run['metrics']['primary_score']})", split=split,
                 pr_auc=by_score[run["metrics"]["primary_score"]]["pr_auc"],
                 f1=by_score[run["metrics"]["primary_score"]]["f1"])
            for name, run in runs.items() for split, by_score in run["metrics"]["splits"].items()]
if resnet_rows:
    display(pd.DataFrame(resnet_rows + vae_rows).pivot(index="model", columns="split", values=["pr_auc", "f1"])
            .reindex(columns=SPLITS, level=1).round(4))
else:
    print("Todavía no hay resultados de ResNet en reports/experiments/resnet")

## 10. Limitaciones

- **Confounding por país:** United_States y China_MotorBike no tienen imágenes sin daño; el VAE nunca vio su
  aspecto normal (sección 5).
- **Resolución:** a 128×128 un D40 típico de Norway mide unos 4×4 px; es esperable un `test_ood` cerca del azar.
- **`test_perspective`:** solo 5 imágenes sin daño; sus comparaciones con ese grupo no son concluyentes.
- **"Sin daño anotado" ≠ carretera perfecta:** los XML vacíos pueden tener daño no anotado.
- **Reproducibilidad:** la evaluación es determinista (z = μ), pero el entrenamiento en GPU no es idéntico bit a bit.